### 프로젝트 : E-Commerce Pay Insight Agent
### 목적: Olist 브라질 이커머스 데이터 탐색 및 데이터마트 설계 근거 도출
### 데이터: Kaggle Olist Brazilian E-Commerce Dataset (9개 테이블)
### 작성일: 2026-03

In [13]:
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

DATA_PATH = Path('../data/raw')

# 주문 메인 테이블
orders    = pd.read_csv(DATA_PATH / 'olist_orders_dataset.csv')
# 결제 테이블
payments  = pd.read_csv(DATA_PATH / 'olist_order_payments_dataset.csv')
# 고객 테이블
customers = pd.read_csv(DATA_PATH / 'olist_customers_dataset.csv')
# 리뷰 테이블
reviews   = pd.read_csv(DATA_PATH / 'olist_order_reviews_dataset.csv')
# 주문 아이템 테이블
items     = pd.read_csv(DATA_PATH / 'olist_order_items_dataset.csv')
# 상품 테이블(카테고리명 포르투갈어)
products  = pd.read_csv(DATA_PATH / 'olist_products_dataset.csv')
# 판매자 테이블
sellers   = pd.read_csv(DATA_PATH / 'olist_sellers_dataset.csv')
# 위치 테이블
geo       = pd.read_csv(DATA_PATH / 'olist_geolocation_dataset.csv')
# 카테고리 번역 테이블(포르투갈어 -> 영어 매핑)
category  = pd.read_csv(DATA_PATH / 'product_category_name_translation.csv')

# 로드 확인
tables = {
    'orders': orders,
    'payments': payments,
    'customers': customers,
    'reviews': reviews,
    'items': items,
    'products': products,
    'sellers': sellers,
    'geo': geo,
    'category': category
}

for name, df in tables.items():
    print(f"{name:12s}: {df.shape[0]:>9,}행  {df.shape[1]:>3}열")

orders      :    99,441행    8열
payments    :   103,886행    5열
customers   :    99,441행    5열
reviews     :    99,224행    7열
items       :   112,650행    7열
products    :    32,951행    9열
sellers     :     3,095행    4열
geo         : 1,000,163행    5열
category    :        71행    2열


### 테이블별 결측치 확인

In [14]:
# 결측치
for name, df in tables.items():
    missing = df.isnull().sum()
    missing = missing[missing > 0]
    if len(missing) > 0:
        print(f"\n[Table : {name}]")
        for col, cnt in missing.items():
            pct = cnt / len(df) * 100
            print(f"  {col:40s}: {cnt:>6,}개 ({pct:.1f}%)")


[Table : orders]
  order_approved_at                       :    160개 (0.2%)
  order_delivered_carrier_date            :  1,783개 (1.8%)
  order_delivered_customer_date           :  2,965개 (3.0%)

[Table : reviews]
  review_comment_title                    : 87,656개 (88.3%)
  review_comment_message                  : 58,247개 (58.7%)

[Table : products]
  product_category_name                   :    610개 (1.9%)
  product_name_lenght                     :    610개 (1.9%)
  product_description_lenght              :    610개 (1.9%)
  product_photos_qty                      :    610개 (1.9%)
  product_weight_g                        :      2개 (0.0%)
  product_length_cm                       :      2개 (0.0%)
  product_height_cm                       :      2개 (0.0%)
  product_width_cm                        :      2개 (0.0%)



[Table : orders]
  * order_approved_at                       : 결제 승인 시각
    * 본 프로젝트에서 사용하지 않는 컬럼
  * order_delivered_carrier_date            : 물품 출고 시각
    * delivery_days 계산 시 결측치 포함 행 제외 예정
  * order_delivered_customer_date           : 고객 수령 시각
    * 배송 완료 전 주문에 해당하며 delivered 상태 필터링하여 제거 예정

[Table : reviews]
  * review_comment_title                    : 리뷰 제목
  * review_comment_message                  : 리뷰 본문 텍스트
    * review_score만 남긴 경우에 해당됨

[Table : products]
  * product_category_name                   : 카테고리명
  * product_name_lenght                     : 상품명 글자 수
  * product_description_lenght              : 상품설명 글자 수
  * product_photos_qty                      : 상품 사진 개수
    * 정보 미등록 상품으로 카테고리명 영문 변환시 'Unknown'으로 매핑 예정
  * product_weight_g                        : 상품 무게(g)
  * product_length_cm                       : 상품 길이(cm)
  * product_height_cm                       : 상품 높이(cm)
  * product_width_cm                        : 상품 너비(cm)
    * 중앙값으로 대체 예정

### 데이터 분포 확인

In [15]:
# Oders - order_status
status_counts = orders['order_status'].value_counts()
status_pct = (status_counts / len(orders) * 100).round(2)

for status, pct in status_pct.items():
    print(f"  {status:20s}: {pct:5.1f}%")

  delivered           :  97.0%
  shipped             :   1.1%
  canceled            :   0.6%
  unavailable         :   0.6%
  invoiced            :   0.3%
  processing          :   0.3%
  created             :   0.0%
  approved            :   0.0%


### 데이터 분포 확인

In [16]:
# Payments - payment_type
pay_counts = payments['payment_type'].value_counts()
pay_pct = (pay_counts / len(payments) * 100).round(2)

for ptype, pct in pay_pct.items():
    print(f"  {ptype:20s}: {pct:5.1f}%")

print(f"\n할부 결제 평균 횟수: {payments['payment_installments'].mean():.1f}회")
print(f"일시불 비중: {(payments['payment_installments'] == 1).mean()*100:.1f}%")
print(f"6회 이상 할부 비중: {(payments['payment_installments'] >= 6).mean()*100:.1f}%")

  credit_card         :  73.9%
  boleto              :  19.0%
  voucher             :   5.6%
  debit_card          :   1.5%
  not_defined         :   0.0%

할부 결제 평균 횟수: 2.9회
일시불 비중: 50.6%
6회 이상 할부 비중: 15.5%


In [17]:
# Reviews - review_score
score_counts = reviews['review_score'].value_counts().sort_index()
score_pct = (score_counts / len(reviews) * 100).round(2)

for score, pct in score_pct.items():
    print(f"  {score}점: {pct:5.1f}%")

print(f"\n평균 리뷰 점수: {reviews['review_score'].mean():.2f}점")
print(f"리뷰 텍스트 존재 비율: {reviews['review_comment_message'].notna().mean()*100:.1f}%")
print(f"낮은 점수(1~2점) 비중: {(reviews['review_score'] <= 2).mean()*100:.1f}%")

  1점:  11.5%
  2점:   3.2%
  3점:   8.2%
  4점:  19.3%
  5점:  57.8%

평균 리뷰 점수: 4.09점
리뷰 텍스트 존재 비율: 41.3%
낮은 점수(1~2점) 비중: 14.7%


### 데이터 전처리

In [18]:
# delivery_days(배송 기간) 계산
date_cols = [
    'order_purchase_timestamp',
    'order_delivered_customer_date',
    'order_estimated_delivery_date'
]
for col in date_cols:
    orders[col] = pd.to_datetime(orders[col])

# delivered 상태만 필터링
delivered = orders[orders['order_status'] == 'delivered'].copy()

# delivery_days 생성 : 고객 수령 날짜 - 주문 버튼 클릭 날짜
delivered['delivery_days'] = (
    delivered['order_delivered_customer_date'] -
    delivered['order_purchase_timestamp']
).dt.days

print(f"  평균 배송일: {delivered['delivery_days'].mean():.1f}일")
print(f"  중앙값: {delivered['delivery_days'].median():.1f}일")
print(f"  최소: {delivered['delivery_days'].min():.0f}일")
print(f"  최대: {delivered['delivery_days'].max():.0f}일")
print(f"  14일 초과(지연) 비중: {(delivered['delivery_days'] > 14).mean()*100:.1f}%")
print(f"  지연 주문 수: {(delivered['delivery_days'] > 14).sum():,}건")

  평균 배송일: 12.1일
  중앙값: 10.0일
  최소: 0일
  최대: 209일
  14일 초과(지연) 비중: 27.3%
  지연 주문 수: 26,377건


In [19]:
# region_group 생성 : customer_state 27개 주를 5개 권역으로 그룹화
region_map = {
    'SP': 'Southeast', 'RJ': 'Southeast', 'MG': 'Southeast', 'ES': 'Southeast',
    'BA': 'Northeast', 'CE': 'Northeast', 'PE': 'Northeast', 'MA': 'Northeast',
    'PB': 'Northeast', 'RN': 'Northeast', 'AL': 'Northeast', 'SE': 'Northeast',
    'PI': 'Northeast', 'RS': 'South', 'PR': 'South', 'SC': 'South',
    'PA': 'North', 'AM': 'North', 'RO': 'North', 'AC': 'North',
    'AP': 'North', 'RR': 'North', 'TO': 'North',
    'GO': 'Central-West', 'MT': 'Central-West', 'MS': 'Central-West', 'DF': 'Central-West'
}

customers['region_group'] = customers['customer_state'].map(region_map).fillna('Other')

region_counts = customers['region_group'].value_counts()
region_pct = (region_counts / len(customers) * 100).round(1)

print('[ 그룹 분포 ]\n')
for region, pct in region_pct.items():
    print(f"  {region:15s}: {pct:5.1f}%")

delivered_with_region = delivered.merge(
    customers[['customer_id', 'customer_state', 'region_group']],
    on='customer_id'
)

print('\n[ 그룹별 평균 배송일 ]\n')
region_delivery = delivered_with_region.groupby('region_group')['delivery_days'].mean().sort_values(ascending=False)
for region, days in region_delivery.items():
    print(f"  {region:15s}: {days:5.1f}일")

[ 그룹 분포 ]

  Southeast      :  68.6%
  South          :  14.2%
  Northeast      :   9.4%
  Central-West   :   5.8%
  North          :   1.9%

[ 그룹별 평균 배송일 ]

  North          :  22.1일
  Northeast      :  19.5일
  Central-West   :  14.6일
  South          :  13.6일
  Southeast      :  10.3일


### 결측치 처리 내역 정리

In [20]:
issues = []

# 1. 결측치
null_delivery = orders['order_delivered_customer_date'].isnull().sum()
issues.append({
    'type': '결측치',
    'location': 'orders.order_delivered_customer_date',
    'count': f'{null_delivery:,}건',
    'action': 'delivered 상태만 필터링하여 처리'
})

null_review = reviews['review_comment_message'].isnull().sum()
issues.append({
    'type': '결측치',
    'location': 'reviews.review_comment_message',
    'count': f'{null_review:,}건',
    'action': '감성 분석 시 notna() 필터링'
})

null_category = products['product_category_name'].isnull().sum()
issues.append({
    'type': '결측치',
    'location': 'products.product_category_name',
    'count': f'{null_category:,}건',
    'action': "'Unknown'으로 대체"
})

# 2. 이상치
extreme_delivery = (delivered['delivery_days'] > 100).sum()
issues.append({
    'type': '이상치',
    'location': 'delivery_days > 100일',
    'count': f'{extreme_delivery:,}건',
    'action': '100일 초과는 이상치로 제거'
})

zero_delivery = (delivered['delivery_days'] == 0).sum()
issues.append({
    'type': '이상치',
    'location': 'delivery_days == 0일',
    'count': f'{zero_delivery:,}건',
    'action': '당일 배송 또는 오류, 1일로 대체'
})

neg_price = items[items['price'] <= 0].shape[0]
issues.append({
    'type': '이상치',
    'location': 'items.price <= 0',
    'count': f'{neg_price:,}건',
    'action': '해당 행 제거'
})

# 3. 중복
dup_orders = orders['order_id'].duplicated().sum()
issues.append({
    'type': '중복',
    'location': 'orders.order_id',
    'count': f'{dup_orders:,}건',
    'action': '중복 제거'
})

# 출력
for i, issue in enumerate(issues, 1):
    print(f"\n{i}. [{issue['type']}] {issue['location']}")
    print(f"   발생 건수: {issue['count']}")
    print(f"   처리 방법: {issue['action']}")

print(f"\n 총 {len(issues)}개 품질 이슈 → ETL DataValidator에서 전부 처리 예정")


1. [결측치] orders.order_delivered_customer_date
   발생 건수: 2,965건
   처리 방법: delivered 상태만 필터링하여 처리

2. [결측치] reviews.review_comment_message
   발생 건수: 58,247건
   처리 방법: 감성 분석 시 notna() 필터링

3. [결측치] products.product_category_name
   발생 건수: 610건
   처리 방법: 'Unknown'으로 대체

4. [이상치] delivery_days > 100일
   발생 건수: 63건
   처리 방법: 100일 초과는 이상치로 제거

5. [이상치] delivery_days == 0일
   발생 건수: 13건
   처리 방법: 당일 배송 또는 오류, 1일로 대체

6. [이상치] items.price <= 0
   발생 건수: 0건
   처리 방법: 해당 행 제거

7. [중복] orders.order_id
   발생 건수: 0건
   처리 방법: 중복 제거

 총 7개 품질 이슈 → ETL DataValidator에서 전부 처리 예정


### EDA 요약

In [21]:
print("""
[ 데이터 규모 ]
  총 주문 수       : 99,441건
  총 결제 건수     : 103,886건
  총 리뷰 수       : 99,224건
  상품 카테고리    : 71종

[ 핵심 인사이트 ]
  1. delivered 비중 97% → converted 컬럼 기준으로 활용 가능
  2. credit_card 73.9%, 6회 이상 할부 15.5% → Installment_Promo 캠페인 근거
  3. 평균 배송일 12.1일, 지연(14일 초과) 27.3% → 배송 지연 분석 시나리오 근거
  4. North 22.1일 vs Southeast 10.3일 → 지역별 배송 격차 11.8일
  5. 저점수(1~2점) 14.7%, 텍스트 리뷰 41.3% → 감성 분석 대상 약 4만 건

[ 품질 이슈 ]
  처리 필요 4건 → Week 3 ETL DataValidator에서 처리 예정
  - order_delivered_customer_date 결측 2,965건
  - review_comment_message 결측 58,247건
  - product_category_name 결측 610건
  - delivery_days 이상치 76건 (>100일 63건, 0일 13건)
""")


[ 데이터 규모 ]
  총 주문 수       : 99,441건
  총 결제 건수     : 103,886건
  총 리뷰 수       : 99,224건
  상품 카테고리    : 71종

[ 핵심 인사이트 ]
  1. delivered 비중 97% → converted 컬럼 기준으로 활용 가능
  2. credit_card 73.9%, 6회 이상 할부 15.5% → Installment_Promo 캠페인 근거
  3. 평균 배송일 12.1일, 지연(14일 초과) 27.3% → 배송 지연 분석 시나리오 근거
  4. North 22.1일 vs Southeast 10.3일 → 지역별 배송 격차 11.8일
  5. 저점수(1~2점) 14.7%, 텍스트 리뷰 41.3% → 감성 분석 대상 약 4만 건

[ 품질 이슈 ]
  처리 필요 4건 → Week 3 ETL DataValidator에서 처리 예정
  - order_delivered_customer_date 결측 2,965건
  - review_comment_message 결측 58,247건
  - product_category_name 결측 610건
  - delivery_days 이상치 76건 (>100일 63건, 0일 13건)

